In [0]:
from datetime import date, timedelta

hoje = date.today()
padrao_inicio = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
padrao_fim = hoje.strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", padrao_inicio, "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", padrao_fim, "Data fim (MM-DD-AAAA)")

In [0]:
from pyspark.sql import functions as F

catalog = "cinedata_medallion"
bronze_schema = f"{catalog}.bronze"
volume_path = f"/Volumes/{catalog}/bronze/landing"

# 1. Database da camada Bronze da criada no notebook CineData_organization
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

# 2-4. Tabelas Bronze a partir dos CSVs
tabelas = ["movies_info", "movies_financials", "movies_metrics",
           "credits_and_tags", "movies_reviews"]

for tabela in tabelas:
    df = (spark.read
          .option("header", True)
          .option("multiLine", True)
          .option("quote", '"')
          .option("escape", '"')
          .csv(f"{volume_path}/*{tabela}*.csv")
          .withColumn("ingestion_datetime", F.current_timestamp()))

    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(f"{bronze_schema}.{tabela}"))

    print(f"{bronze_schema}.{tabela}: {df.count()} linhas lidas")

In [0]:
import requests
from datetime import datetime
from pyspark.sql import functions as F

catalog = "cinedata_medallion"
tabela_destino = f"{catalog}.bronze.tb_cotacao_dolar"

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

# valida o formato MM-DD-AAAA antes de chamar a API
for d in (data_inicio_formatada, data_fim_formatada):
    datetime.strptime(d, "%m-%d-%Y")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
cotacoes = resposta.json()["value"]

print(f"Período: {data_inicio_formatada} a {data_fim_formatada} | {len(cotacoes)} cotações retornadas")

if cotacoes:
    df = (spark.createDataFrame(cotacoes)
          .select("dataHoraCotacao", "cotacaoCompra")
          .withColumn("ingestion_datetime", F.current_timestamp()))

    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(tabela_destino))

    display(spark.table(tabela_destino).orderBy(F.desc("ingestion_datetime")))
else:
    print("Nenhuma cotação no período (fim de semana/feriado?). Nada gravado.")